# SimCLR Colab Run

This notebook runs the SimCLR contrastive learning pipeline: pretraining → fine-tuning → evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUN_NAME = f'simclr_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_ZIP =', DATASET_ZIP)
print('DRIVE_RUN_DIR =', DRIVE_RUN_DIR)

In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

## Install Dependencies

In [ ]:
%cd "$PROJECT_DIR"
!pip install -q openpyxl scikit-learn pandas matplotlib pillow opencv-python

## Setup Dataset

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./dataset'):
    shutil.rmtree('./dataset')

!unzip -q "$DATASET_ZIP" -d .
!python data/setup.py

## Link Output to Drive

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./output'):
    shutil.rmtree('./output')

os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.symlink(DRIVE_RUN_DIR, './output')
print(f'Linked ./output -> {DRIVE_RUN_DIR}')

In [ ]:
%cd "$PROJECT_DIR"

# Fix the np.prod bug in cae_model.py
with open('models/cae_model.py', 'r') as f:
    content = f.read()

# Fix line 131 - wrap np.prod with int()
content = content.replace(
    'x1 = Dense(np.prod(shape_before_flattening))(encoder_output)',
    'x1 = Dense(int(np.prod(shape_before_flattening)))(encoder_output)'
)

with open('models/cae_model.py', 'w') as f:
    f.write(content)

print('✓ Fixed cae_model.py np.prod bug')

## Fix Model Bug

Fix the np.prod() bug in cae_model.py that affects all models using this architecture.

In [ ]:
# Fix reset_states() → reset_state() bug in simclr_model.py (Keras 3.x API change)
simclr_model_path = "/content/SSL_Prostate_Cancer_Grading/models/simclr_model.py"

with open(simclr_model_path, 'r') as f:
    content = f.read()

# Fix the method name (Keras 3.x uses singular reset_state, not reset_states)
content = content.replace('reset_states()', 'reset_state()')

with open(simclr_model_path, 'w') as f:
    f.write(content)

print("✓ Fixed reset_states() → reset_state() in simclr_model.py")

In [ ]:
# Fix UnboundLocalError in generator.py (for contrastive learning without labels)
generator_path = "/content/SSL_Prostate_Cancer_Grading/data/generator.py"

with open(generator_path, 'r') as f:
    content = f.read()

# Add default label initialization
old_code = """        if self.mode == 'mClass':
            label = self.dict_classes[df_row["group"]][self.outputs]
        if self.mode == 'mLabel':"""

new_code = """        # Default label to None (for contrastive learning like SimCLR)
        label = None
        
        if self.mode == 'mClass':
            label = self.dict_classes[df_row["group"]][self.outputs]
        if self.mode == 'mLabel':"""

content = content.replace(old_code, new_code)

with open(generator_path, 'w') as f:
    f.write(content)

print("✓ Fixed UnboundLocalError in generator.py")

## Apply Patches for SimCLR

Add CLI arguments to pretrain_simclr.py and finetune_simclr.py.

In [ ]:
%cd "$PROJECT_DIR"
import re

# Patch pretrain_simclr.py
pretrain_path = 'training/simclr/pretrain_simclr.py'
with open(pretrain_path, 'r') as f:
    content = f.read()

if 'argparse' not in content:
    # Add argparse import
    content = content.replace(
        'import sys',
        'import sys\nimport argparse'
    )
    
    # Add argument parser before CONFIGURATION
    parser_code = '''
def parse_args():
    parser = argparse.ArgumentParser(description="SimCLR contrastive learning pretraining")
    parser.add_argument('--epochs', type=int, default=30, help='Number of training epochs')
    parser.add_argument('--batch_size', type=int, default=64, help='Batch size')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--temperature', type=float, default=0.5, help='Temperature for NT-Xent loss')
    return parser.parse_args()

args = parse_args()

'''
    # Insert before CONFIGURATION
    content = re.sub(
        r'(# ={70,}\n# CONFIGURATION\n# ={70,})',
        parser_code + r'\1',
        content
    )
    
    # Replace hardcoded values
    content = content.replace('BATCH_SIZE = 64', 'BATCH_SIZE = args.batch_size')
    content = content.replace('EPOCHS = 30', 'EPOCHS = args.epochs')
    content = content.replace('LEARNING_RATE = 0.001', 'LEARNING_RATE = args.lr')
    content = content.replace('TEMPERATURE = 0.5', 'TEMPERATURE = args.temperature')
    
    with open(pretrain_path, 'w') as f:
        f.write(content)
    print('✓ Patched pretrain_simclr.py with CLI arguments')
else:
    print('✓ pretrain_simclr.py already patched')

# Patch finetune_simclr.py
finetune_path = 'training/simclr/finetune_simclr.py'
with open(finetune_path, 'r') as f:
    content = f.read()

if 'argparse' not in content:
    # Add argparse import
    content = content.replace(
        'import sys',
        'import sys\nimport argparse'
    )
    
    # Add argument parser
    parser_code = '''
def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune SimCLR encoder")
    parser.add_argument('--epochs_stage1', type=int, default=50, help='Stage 1 epochs (frozen encoder)')
    parser.add_argument('--epochs_stage2', type=int, default=0, help='Stage 2 epochs (unfrozen encoder)')
    parser.add_argument('--batch_size', type=int, default=8, help='Batch size')
    return parser.parse_args()

args = parse_args()

'''
    # Insert before CONFIGURATION
    content = re.sub(
        r'(# ={70,}\n# CONFIGURATION\n# ={70,})',
        parser_code + r'\1',
        content
    )
    
    # Replace hardcoded values
    content = content.replace('BATCH_SIZE = 8', 'BATCH_SIZE = args.batch_size')
    content = content.replace('EPOCHS_STAGE_1 = 50', 'EPOCHS_STAGE_1 = args.epochs_stage1')
    content = content.replace('EPOCHS_STAGE_2 = 0', 'EPOCHS_STAGE_2 = args.epochs_stage2')
    
    with open(finetune_path, 'w') as f:
        f.write(content)
    print('✓ Patched finetune_simclr.py with CLI arguments')
else:
    print('✓ finetune_simclr.py already patched')

In [ ]:
# Fix TensorFlow layout optimizer and learning rate access issues
pretrain_path = 'training/simclr/pretrain_simclr.py'

with open(pretrain_path, 'r') as f:
    content = f.read()

# Fix 1: Disable TensorFlow layout optimizer (add env vars after imports)
if 'TF_DISABLE_LAYOUT_OPTIMIZER' not in content:
    import_section = """import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime"""
    
    fixed_import = """# Disable TensorFlow layout optimizer to prevent NHWC/NCHW transpose errors
import os
os.environ['TF_DISABLE_LAYOUT_OPTIMIZER'] = '1'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime"""
    
    content = content.replace(import_section, fixed_import)
    print("✓ Added TensorFlow optimizer disabling")

# Fix 2: Fix learning rate access for Keras 3.x
old_lr_line = "    current_lr = optimizer.learning_rate(epoch * len(train_generator)).numpy()"
new_lr_line = """    # Get current learning rate (Keras 3.x compatible)
    if hasattr(optimizer.learning_rate, '__call__'):
        current_lr = optimizer.learning_rate(epoch * len(train_generator)).numpy()
    else:
        current_lr = optimizer.learning_rate.numpy() if hasattr(optimizer.learning_rate, 'numpy') else float(optimizer.learning_rate)"""

if old_lr_line in content:
    content = content.replace(old_lr_line, new_lr_line)
    print("✓ Fixed learning rate access")

with open(pretrain_path, 'w') as f:
    f.write(content)

print("✓ All TensorFlow/Keras 3.x compatibility fixes applied")

In [ ]:
# Fix checkpoint filename for Keras 3.x (must end in .weights.h5)
pretrain_path = 'training/simclr/pretrain_simclr.py'

with open(pretrain_path, 'r') as f:
    content = f.read()

# Fix checkpoint filenames to use .weights.h5 extension
content = content.replace(
    'checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_model_epoch_{epoch+1}.h5")',
    'checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_model_epoch_{epoch+1}.weights.h5")'
)

content = content.replace(
    'checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.h5")',
    'checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.weights.h5")'
)

# Also fix the final encoder weights save
content = content.replace(
    'encoder.save_weights(os.path.join(OUTPUT_DIR, "encoder_weights.h5"))',
    'encoder.save_weights(os.path.join(OUTPUT_DIR, "encoder_weights.weights.h5"))'
)

# Fix the model save (use keras format instead of h5)
content = content.replace(
    'simclr_model.save(os.path.join(OUTPUT_DIR, "simclr_model.h5"))',
    'simclr_model.save(os.path.join(OUTPUT_DIR, "simclr_model.keras"))'
)

with open(pretrain_path, 'w') as f:
    f.write(content)

print("✓ Fixed checkpoint filenames for Keras 3.x (.weights.h5 extension)")

## Pretrain SimCLR

Train the encoder using contrastive learning (NT-Xent loss on augmented pairs).

In [ ]:
%cd "$PROJECT_DIR"
EPOCHS = 50         # Increase from default 30 for better representations
BATCH_SIZE = 128    # T4 can handle 128, A100 can try 256
LR = 0.001          # Default learning rate
TEMPERATURE = 0.5   # Temperature for NT-Xent loss

cmd = f'python training/simclr/pretrain_simclr.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --temperature {TEMPERATURE}'
print(cmd)
!$cmd

## Fine-Tune SimCLR

Load pretrained encoder and train classification head. Captures both frozen-encoder (Stage 1) and optional end-to-end fine-tuning (Stage 2).

In [ ]:
%cd "$PROJECT_DIR"

FINETUNE_BATCH_SIZE = 32    # T4 can handle 32
EPOCHS_STAGE_1 = 50         # Head-only training
EPOCHS_STAGE_2 = 20         # End-to-end fine-tuning (set to 0 to skip)

cmd = (
    f'python training/simclr/finetune_simclr.py'
    f' --epochs_stage1 {EPOCHS_STAGE_1}'
    f' --epochs_stage2 {EPOCHS_STAGE_2}'
    f' --batch_size {FINETUNE_BATCH_SIZE}'
)
print(cmd)
!$cmd

## Evaluate SimCLR

Generate evaluation metrics on the test set.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluation/simclr/eval_simclr.py

## Sync Runtime State to Drive

In [ ]:
%cd "$PROJECT_DIR"

import shutil
from pathlib import Path

sync_root = Path(DRIVE_RUN_DIR) / "runtime_sync"
dataset_sync = sync_root / "dataset"
notebook_sync = sync_root / "notebook"

dataset_sync.mkdir(parents=True, exist_ok=True)
notebook_sync.mkdir(parents=True, exist_ok=True)

# Copy notebook if it exists in project dir (optional)
if Path("run_simclr_colab.ipynb").exists():
    shutil.copy2("run_simclr_colab.ipynb", notebook_sync / "run_simclr_colab.ipynb")

for name in ["Train.csv", "Test.csv", "TrainSplit.csv", "Val.csv"]:
    src = Path("dataset") / name
    if src.exists():
        shutil.copy2(src, dataset_sync / name)

os.system(f'git rev-parse HEAD > "{sync_root / "commit.txt"}"')
os.system(f'find ./output -maxdepth 4 -type f | sort > "{sync_root / "output_manifest.txt"}"')

print(f"Synced runtime artifacts to: {sync_root}")

In [ ]:
%cd "$PROJECT_DIR"
!echo "Run directory: $DRIVE_RUN_DIR"
!find ./output -maxdepth 3 -type f | sort | tail -n 30